# LightOnOCR-2 Magyar Fine-tuning

Ez a notebook fine-tunolja a LightOnOCR-2 modellt magyar nyelvű dokumentumokra.

**Futtatás előtt:** Runtime → Change runtime type → T4 GPU

In [ ]:
# 1. Telepítés
!pip install -q transformers peft datasets accelerate bitsandbytes pillow

In [ ]:
# 2. Tanító adatok generálása
import json
import random
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

HUNGARIAN_WORDS = [
    "őr", "őriz", "ők", "ősz", "ősi", "őszinte", "őrült",
    "erő", "idő", "mező", "tető", "fő", "nő", "bő", "hő",
    "belső", "külső", "felső", "alsó", "utolsó", "első",
    "költő", "festő", "vezető", "Győr", "tükörfúrógép",
    "Csatornadíj", "vízdíj", "díj", "dőlt", "dől",
    "űr", "űrlap", "gyűrű", "tűz", "fűz", "gyűjt",
    "hűtő", "hűvös", "hűség", "szürke", "szűk",
    "halványszürke", "fizetendő", "összeg", "adószám",
    "öüóőúéáűí", "ÖÜÓŐÚÉÁŰÍ", "Árvíztűrő",
]

def generate_text():
    words = random.sample(HUNGARIAN_WORDS, min(8, len(HUNGARIAN_WORDS)))
    lines = [" ".join(words)]
    lines.append(f"Fizetendő összeg: {random.randint(1,99)} {random.randint(100,999):03d} Ft")
    lines.append(f"Adószám: {random.randint(10000000,99999999)}-{random.randint(1,2)}-{random.randint(10,99)}")
    lines.append("öüóőúéáűí - ÖÜÓŐÚÉÁŰÍ")
    lines.append("Halványszürke szöveg, dőlt betűk")
    return "\n".join(lines)

def render_text(text, width=600, font_size=20):
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", font_size)
    except:
        font = ImageFont.load_default()
    lines = text.split("\n")
    height = len(lines) * (font_size + 8) + 40
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    y = 20
    for line in lines:
        draw.text((20, y), line, fill="black", font=font)
        y += font_size + 8
    return img

# Generálás
Path("training_data/images").mkdir(parents=True, exist_ok=True)
annotations = []

NUM_SAMPLES = 300  # Növelhető
for i in range(NUM_SAMPLES):
    text = generate_text()
    img = render_text(text, font_size=random.choice([18, 20, 22, 24]))
    img.save(f"training_data/images/{i:05d}.png")
    annotations.append({"image": f"{i:05d}.png", "text": text})

with open("training_data/annotations.jsonl", "w") as f:
    for a in annotations:
        f.write(json.dumps(a, ensure_ascii=False) + "\n")

print(f"Generated {NUM_SAMPLES} training samples")

In [ ]:
# 3. Modell betöltése
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "lightonai/LightOnOCR-2-1B-base"

# 4-bit kvantálás memória spóroláshoz
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

print(f"Model loaded: {MODEL_ID}")

In [ ]:
# 4. LoRA konfiguráció
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 5. Dataset előkészítése
from datasets import Dataset
from PIL import Image

def load_data():
    data = []
    with open("training_data/annotations.jsonl") as f:
        for line in f:
            entry = json.loads(line)
            data.append({
                "image_path": f"training_data/images/{entry['image']}",
                "text": entry["text"]
            })
    return Dataset.from_list(data)

def process_example(example):
    image = Image.open(example["image_path"]).convert("RGB")
    conversation = [
        {"role": "user", "content": [{"type": "image"}]},
        {"role": "assistant", "content": example["text"]}
    ]
    inputs = processor.apply_chat_template(
        conversation,
        add_generation_prompt=False,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    image_inputs = processor.image_processor(image, return_tensors="pt")
    inputs["pixel_values"] = image_inputs["pixel_values"]
    inputs["labels"] = inputs["input_ids"].clone()
    return {k: v.squeeze(0) for k, v in inputs.items()}

dataset = load_data()
processed_dataset = dataset.map(process_example, remove_columns=dataset.column_names)
print(f"Dataset size: {len(processed_dataset)}")

In [ ]:
# 6. Training
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./lighton-hun-lora",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    logging_steps=20,
    save_steps=100,
    bf16=True,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset,
)

trainer.train()

In [ ]:
# 7. Mentés és LoRA merge
trainer.save_model("./lighton-hun-lora")

# Merge LoRA weights
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./lighton-hun-merged")
processor.save_pretrained("./lighton-hun-merged")

print("Model saved to ./lighton-hun-merged")

In [ ]:
# 8. Teszt
from PIL import Image

test_img = Image.open("training_data/images/00000.png")
inputs = processor(test_img, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = merged_model.generate(**inputs, max_new_tokens=200)

result = processor.decode(outputs[0], skip_special_tokens=True)
print("OCR result:")
print(result)

In [ ]:
# 9. Letöltés (zip)
!zip -r lighton-hun-merged.zip lighton-hun-merged/
from google.colab import files
files.download("lighton-hun-merged.zip")

print("\nKövetkező lépés Mac-en:")
print("  unzip lighton-hun-merged.zip")
print("  mlx_vlm convert --hf-path lighton-hun-merged --mlx-path lighton-hun-mlx -q --q-bits 4")